# Brain Tumor Detection from MRI Images
### Data Science Honours Project — Streamlit Web Application Harness

This notebook runs the interactive Streamlit application directly inside Google Colab and exposes it via a public URL using **LocalTunnel** or **Cloudflare Tunnel**.

## 1. Install Dependencies

In [ ]:
# Install Streamlit and tunneling utility
!pip install -q streamlit pyngrok
!npm install -g localtunnel

## 2. Upload or Extract Project Files
Ensure your saved files (`custom_cnn.pth`, `resnet18.pth`, `mobilenetv3.pth`, `class_names.json`, `model_comparison.csv`) are placed in the Colab working directory (`/content`).

In [ ]:
import os
import zipfile

# If your files are compressed in a zip archive (e.g. brain_tumor_files.zip):
zip_filename = "brain_tumor_files.zip"
if os.path.exists(zip_filename):
    with zipfile.ZipFile(zip_filename, 'r') as zf:
        zf.extractall(".")
    print("Successfully extracted archive.")
else:
    print("No zip archive found. You can upload individual files or your zip archive now using the Files sidebar.")

## 3. Verify Required Files
Check that all five required assets are present.

In [ ]:
import os

required_files = [
    "custom_cnn.pth",
    "resnet18.pth",
    "mobilenetv3.pth",
    "class_names.json",
    "model_comparison.csv"
]

print("=== File Verification Status ===")
for f in required_files:
    status = "FOUND [OK]" if os.path.exists(f) else "MISSING [!]"
    print(f"{status:<15} {f}")

## 4. Create `app.py`
Run this cell to write the complete application code.

In [ ]:
%%writefile app.py
"""
Brain Tumor Detection from MRI Images - Streamlit Web Application
"""
import os
import json
import numpy as np
import pandas as pd
import streamlit as st
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import matplotlib.pyplot as plt

# =========================================================
# 1. MODEL ARCHITECTURES & DEFINITIONS
# =========================================================

class CustomCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(CustomCNN, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

def build_model(model_name: str, num_classes: int = 4) -> nn.Module:
    name = model_name.strip().lower()
    if "resnet18" in name:
        model = models.resnet18(weights=None)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model
    elif "mobilenetv3" in name or "mobilenet" in name:
        model = models.mobilenet_v3_large(weights=None)
        model.classifier[3] = nn.Linear(model.classifier[3].in_features, num_classes)
        return model
    elif "custom" in name or "cnn" in name:
        return CustomCNN(num_classes=num_classes)
    else:
        raise ValueError(f"Unsupported model: {model_name}")

def clean_state_dict(checkpoint):
    if isinstance(checkpoint, nn.Module):
        return checkpoint.state_dict()
    elif isinstance(checkpoint, dict):
        if "state_dict" in checkpoint:
            state_dict = checkpoint["state_dict"]
        elif "model_state_dict" in checkpoint:
            state_dict = checkpoint["model_state_dict"]
        elif "model" in checkpoint:
            state_dict = checkpoint["model"]
        else:
            state_dict = checkpoint
    else:
        state_dict = checkpoint

    cleaned = {}
    for k, v in state_dict.items():
        new_key = k.replace("module.", "").replace("model.", "")
        cleaned[new_key] = v
    return cleaned

def load_model_weights(model: nn.Module, weights_path: str, device: str = "cpu"):
    if not os.path.exists(weights_path):
        return model, f"Weight file not found: {weights_path}"
    try:
        checkpoint = torch.load(weights_path, map_location=torch.device(device))
        if isinstance(checkpoint, nn.Module):
            model = checkpoint
            model.to(device)
            model.eval()
            return model, "Loaded successfully as serialized nn.Module."

        cleaned_dict = clean_state_dict(checkpoint)
        try:
            model.load_state_dict(cleaned_dict, strict=True)
            model.to(device)
            model.eval()
            return model, "Weights loaded successfully (strict match)."
        except RuntimeError:
            missing, unexpected = model.load_state_dict(cleaned_dict, strict=False)
            model.to(device)
            model.eval()
            return model, f"Weights loaded (non-strict). Missing: {len(missing)}, Unexpected: {len(unexpected)}."
    except Exception as e:
        return model, f"Error loading weights: {str(e)}"

def get_preprocessing_transforms(image_size=(224, 224)):
    return transforms.Compose([
        transforms.Resize(image_size),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

def safe_open_image(file):
    img = Image.open(file)
    if img.mode in ('RGBA', 'LA') or (img.mode == 'P' and 'transparency' in img.info):
        bg = Image.new('RGB', img.size, (255, 255, 255))
        rgba = img.convert('RGBA')
        bg.paste(rgba, mask=rgba.split()[3])
        return bg
    return img.convert('RGB')

# =========================================================
# 2. GRAD-CAM IMPLEMENTATION
# =========================================================
class GradCAM:
    def __init__(self, model: nn.Module, target_layer: nn.Module):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        self.handles = []
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(module, input, output):
            self.activations = output.detach()
        def backward_hook(module, grad_input, grad_output):
            if grad_output and len(grad_output) > 0 and grad_output[0] is not None:
                self.gradients = grad_output[0].detach()
        h_fwd = self.target_layer.register_forward_hook(forward_hook)
        if hasattr(self.target_layer, "register_full_backward_hook"):
            h_bwd = self.target_layer.register_full_backward_hook(backward_hook)
        else:
            h_bwd = self.target_layer.register_backward_hook(backward_hook)
        self.handles.extend([h_fwd, h_bwd])

    def generate_cam(self, input_tensor: torch.Tensor, target_class: int = None):
        self.model.eval()
        self.model.zero_grad()
        if input_tensor.dim() == 3:
            input_tensor = input_tensor.unsqueeze(0)
        output = self.model(input_tensor)
        if target_class is None:
            target_class = int(torch.argmax(output, dim=1).item())
        score = output[0, target_class]
        score.backward(retain_graph=True)
        if self.gradients is None or self.activations is None:
            raise RuntimeError("Grad-CAM could not intercept hooks.")
        weights = torch.mean(self.gradients, dim=[0, 2, 3])
        act = self.activations.clone()
        for i in range(act.shape[1]):
            act[:, i, :, :] *= weights[i]
        cam = torch.sum(act, dim=1).squeeze()
        cam = torch.clamp(cam, min=0.0)
        cam_np = cam.cpu().numpy()
        cam_min, cam_max = np.min(cam_np), np.max(cam_np)
        if cam_max > cam_min:
            cam_norm = (cam_np - cam_min) / (cam_max - cam_min)
        else:
            cam_norm = np.zeros_like(cam_np)
        return cam_norm, target_class

    def remove_hooks(self):
        for handle in self.handles:
            handle.remove()
        self.handles.clear()

def get_target_layer_for_model(model: nn.Module, model_name: str):
    name = model_name.strip().lower()
    if "resnet" in name:
        return model.layer4[-1]
    elif "mobilenet" in name:
        return model.features[-1]
    elif "custom" in name or "cnn" in name:
        for layer in reversed(model.features):
            if isinstance(layer, nn.Conv2d):
                return layer
        return model.features[-1]
    raise ValueError(f"Could not locate target layer for {model_name}.")

def apply_gradcam_overlay(original_pil, cam_mask, alpha=0.45, colormap_name="jet"):
    img_rgb = original_pil.convert("RGB").resize((224, 224))
    img_np = np.array(img_rgb, dtype=np.float32) / 255.0
    cam_pil = Image.fromarray(cam_mask.astype(np.float32))
    cam_resized = np.array(cam_pil.resize((224, 224), Image.Resampling.BICUBIC))
    cam_resized = np.clip(cam_resized, 0.0, 1.0)
    try:
        cmap = plt.get_cmap(colormap_name)
    except Exception:
        cmap = plt.get_cmap("jet")
    heatmap_colored = cmap(cam_resized)[:, :, :3]
    blended = (1.0 - alpha) * img_np + alpha * heatmap_colored
    blended = np.clip(blended, 0.0, 1.0)
    return Image.fromarray((heatmap_colored * 255).astype(np.uint8)), Image.fromarray((blended * 255).astype(np.uint8))

# =========================================================
# 3. STREAMLIT APP
# =========================================================
st.set_page_config(page_title="Brain Tumor Detection | Data Science Honours", page_icon="🧠", layout="wide")

@st.cache_data
def load_class_names(json_path="class_names.json"):
    if os.path.exists(json_path):
        try:
            with open(json_path, "r") as f:
                data = json.load(f)
                if isinstance(data, list): return data
                elif isinstance(data, dict): return [data[str(i)] for i in range(len(data))]
        except Exception: pass
    return ["glioma", "meningioma", "notumor", "pituitary"]

@st.cache_resource
def load_model_cached(model_name, num_classes):
    fn_map = {"Custom CNN": "custom_cnn.pth", "ResNet18": "resnet18.pth", "MobileNetV3": "mobilenetv3.pth"}
    fname = fn_map.get(model_name, "custom_cnn.pth")
    model = build_model(model_name, num_classes=num_classes)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if os.path.exists(fname):
        model, msg = load_model_weights(model, fname, device=device)
        return model, ("successfully" in msg.lower() or "strict" in msg.lower()), msg, device
    return model, False, f"File {fname} not found.", device

classes = load_class_names()
num_classes = len(classes)

st.sidebar.title("🧠 NeuroScan AI")
nav_choice = st.sidebar.radio("Navigation Menu", ["🩺 MRI Inference", "🔍 Grad-CAM Explainability", "📊 Model Comparison", "ℹ️ Project Information"])

def medical_disclaimer():
    st.warning("⚠️ Academic Demonstration Notice: Developed strictly for an undergraduate Honours project. NOT an accredited diagnostic tool. Outputs must not be used for clinical decisions.")

if nav_choice == "🩺 MRI Inference":
    st.header("Brain Tumor MRI Classification")
    medical_disclaimer()
    c_ctrl, c_view = st.columns(2)
    with c_ctrl:
        model_choice = st.selectbox("Select Model", ["ResNet18", "MobileNetV3", "Custom CNN"])
        model, is_loaded, status_msg, device = load_model_cached(model_choice, num_classes)
        if is_loaded: st.success(f"Loaded {model_choice} ({status_msg})")
        else: st.warning(f"Uninitialized weights for {model_choice}: {status_msg}")
        uploaded_file = st.file_uploader("Upload Brain MRI Scan (All image formats accepted)", type=None)
    with c_view:
        if uploaded_file:
            pil_img = safe_open_image(uploaded_file)
            st.image(pil_img, caption="Uploaded Scan", use_container_width=True)
        else: pil_img = None; st.info("Awaiting image upload.")
    if pil_img:
        st.markdown("---")
        transform = get_preprocessing_transforms()
        t_in = transform(pil_img).unsqueeze(0).to(device)
        model.eval()
        with torch.no_grad():
            probs = F.softmax(model(t_in), dim=1).squeeze().cpu().numpy()
        pred_idx = int(np.argmax(probs))
        pred_class = classes[pred_idx]
        conf = float(probs[pred_idx]) * 100.0
        r1, r2 = st.columns(2)
        with r1:
            st.metric("Predicted Class", pred_class.upper(), f"{conf:.2f}% Confidence")
        with r2:
            p_df = pd.DataFrame({"Class": [c.capitalize() for c in classes], "Probability (%)": probs * 100}).sort_values(by="Probability (%)", ascending=False)
            st.bar_chart(p_df.set_index("Class"))

elif nav_choice == "🔍 Grad-CAM Explainability":
    st.header("Grad-CAM Model Explainability")
    medical_disclaimer()
    cam_model_choice = st.selectbox("Select Model", ["ResNet18", "MobileNetV3", "Custom CNN"], key="cam_m")
    model, is_loaded, status_msg, device = load_model_cached(cam_model_choice, num_classes)
    up_cam = st.file_uploader("Upload MRI for Saliency (All image formats accepted)", type=None, key="cam_f")
    alpha = st.slider("Overlay Opacity (Alpha)", 0.1, 0.9, 0.45, 0.05)
    cmap = st.selectbox("Colormap", ["jet", "viridis", "inferno", "plasma"])
    if up_cam:
        p_cam = safe_open_image(up_cam)
        target_layer = get_target_layer_for_model(model, cam_model_choice)
        engine = GradCAM(model, target_layer)
        t_in = get_preprocessing_transforms()(p_cam).unsqueeze(0).to(device)
        mask, pred_cls = engine.generate_cam(t_in)
        engine.remove_hooks()
        h_img, o_img = apply_gradcam_overlay(p_cam, mask, alpha=alpha, colormap_name=cmap)
        c1, c2, c3 = st.columns(3)
        with c1: st.image(p_cam.resize((224, 224)), caption="Original (224x224)", use_container_width=True)
        with c2: st.image(h_img, caption=f"Heatmap ({cmap})", use_container_width=True)
        with c3: st.image(o_img, caption="Grad-CAM Overlay", use_container_width=True)
        st.success(f"Target Class Attributed: {classes[pred_cls].capitalize()}")

elif nav_choice == "📊 Model Comparison":
    st.header("Comparative Model Performance Dashboard")
    medical_disclaimer()
    if os.path.exists("model_comparison.csv"):
        df = pd.read_csv("model_comparison.csv")
        st.dataframe(df, use_container_width=True)
        if "Accuracy" in df.columns:
            best = df.loc[df["Accuracy"].astype(float).idxmax()]
            st.success(f"🏆 Top Performing Model: **{best['Model']}** with {float(best['Accuracy'])*100:.2f}% Accuracy")
    else:
        st.warning("model_comparison.csv not found.")

elif nav_choice == "ℹ️ Project Information":
    st.header("Project Overview")
    medical_disclaimer()
    st.markdown("""
    - **Dataset:** `masoudnickparvar/brain-tumor-mri-dataset` (7,023 scans)
    - **Classes:** Glioma, Meningioma, Pituitary, No Tumor
    - **Architectures:** ResNet18, MobileNetV3, Custom CNN
    - **Input Resolution:** 224x224 with ImageNet normalization
    """)


## 5. Expose Public URL and Launch Application

In [ ]:
# Get Colab external IP (Password for LocalTunnel)
import urllib
print("------------------------------------------------------------------------")
print("YOUR LOCALTUNNEL PASSWORD (IP ADDRESS) IS:")
print(urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip())
print("------------------------------------------------------------------------\n")

# Launch Streamlit in the background and expose via port 8501
!streamlit run app.py & npx localtunnel --port 8501